In [1]:
from flask import Flask, request, render_template_string, redirect
import sqlite3
import numpy as np

app = Flask(__name__)

# Minimal HTML for the grading interface
HTML = """
<!DOCTYPE html>
<html>
<body style="background: #222; color: white; font-family: sans-serif; text-align: center; padding-top: 50px;">
    <h1>Data Grader</h1>
    
    {% if rowid %}
        <p>The Model Guessed: <strong style="color: #ffeb3b; font-size: 24px;">{{ guess }}</strong></p>
        
        <canvas id="c" width="28" height="28" style="border: 2px solid white; background: black; transform: scale(5); image-rendering: pixelated; margin: 60px;"></canvas>
        
        <br>
        <form action="/grade" method="POST" style="margin-top: 20px;">
            <input type="hidden" name="rowid" value="{{ rowid }}">
            <label style="font-size: 18px;">Correct Letter: </label>
            <input type="text" name="correct_label" maxlength="1" required autofocus autocomplete="off"
                   style="text-transform: uppercase; font-size: 20px; width: 40px; text-align: center;">
            <button type="submit" style="padding: 10px 20px; font-size: 16px; cursor: pointer;">Save & Next</button>
        </form>
        
        <form action="/discard" method="POST" style="margin-top: 30px;">
            <input type="hidden" name="rowid" value="{{ rowid }}">
            <button type="submit" style="padding: 8px 16px; font-size: 14px; background: #552222; color: white; border: none; cursor: pointer;">Discard / Junk Data</button>
        </form>

        <script>
            // Reconstruct the 28x28 image from the flattened float array
            const canvas = document.getElementById('c');
            const ctx = canvas.getContext('2d');
            const pixels = {{ pixels }};
            const imgData = ctx.createImageData(28, 28);
            
            for (let i = 0; i < pixels.length; i++) {
                const val = pixels[i] * 255; // Un-normalize back to 0-255
                imgData.data[i*4] = val;     // Red
                imgData.data[i*4+1] = val;   // Green
                imgData.data[i*4+2] = val;   // Blue
                imgData.data[i*4+3] = 255;   // Alpha (fully opaque)
            }
            ctx.putImageData(imgData, 0, 0);
        </script>
    {% else %}
        <h2>🎉 All caught up!</h2>
        <p>The drawing_logs queue is empty.</p>
    {% endif %}
</body>
</html>
"""

def init_training_table(db_path='brain.db'):
    """Ensures we have a table to store the clean, graded data."""
    conn = sqlite3.connect(db_path)
    # image_blob is the 28x28 float array, label is the confirmed correct letter
    conn.execute("CREATE TABLE IF NOT EXISTS training_data (image_blob BLOB, label TEXT)")
    conn.commit()
    conn.close()

@app.route('/')
def index():
    conn = sqlite3.connect('brain.db')
    # Grab one row from the logs using SQLite's built-in rowid
    row = conn.execute("SELECT rowid, image_blob, guess FROM drawing_logs LIMIT 1").fetchone()
    conn.close()

    if row:
        rowid, blob, guess = row
        # Convert the binary blob back into a Python list so Jinja can pass it to JavaScript
        pixels = np.frombuffer(blob, dtype=np.float32).tolist()
        return render_template_string(HTML, rowid=rowid, guess=guess, pixels=pixels)
    else:
        return render_template_string(HTML, rowid=None)

@app.route('/grade', methods=['POST'])
def grade():
    rowid = request.form['rowid']
    label = request.form['correct_label'].upper()

    conn = sqlite3.connect('brain.db')
    # Fetch the original blob
    row = conn.execute("SELECT image_blob FROM drawing_logs WHERE rowid=?", (rowid,)).fetchone()
    if row:
        # 1. Insert into the new training_data table
        conn.execute("INSERT INTO training_data VALUES (?, ?)", (row[0], label))
        # 2. Delete from the ungraded queue
        conn.execute("DELETE FROM drawing_logs WHERE rowid=?", (rowid,))
        conn.commit()
    conn.close()
    
    return redirect('/')

@app.route('/discard', methods=['POST'])
def discard():
    """Allows you to throw away squiggles or mistakes without polluting the training data."""
    rowid = request.form['rowid']
    conn = sqlite3.connect('brain.db')
    conn.execute("DELETE FROM drawing_logs WHERE rowid=?", (rowid,))
    conn.commit()
    conn.close()
    
    return redirect('/')

if __name__ == '__main__':
    init_training_table()
    # Run on a different port so you can have both apps running simultaneously
    app.run(port=5001)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [07/Apr/2026 10:46:03] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:46:03] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [07/Apr/2026 10:46:12] "POST /discard HTTP/1.1" 302 -
127.0.0.1 - - [07/Apr/2026 10:46:12] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:46:15] "POST /grade HTTP/1.1" 302 -
127.0.0.1 - - [07/Apr/2026 10:46:15] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:46:17] "POST /discard HTTP/1.1" 302 -
127.0.0.1 - - [07/Apr/2026 10:46:17] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:46:20] "POST /grade HTTP/1.1" 302 -
127.0.0.1 - - [07/Apr/2026 10:46:20] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:46:22] "POST /grade HTTP/1.1" 302 -
127.0.0.1 - - [07/Apr/2026 10:46:22] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:46:24] "POST /grade HTTP/1.1" 302 -
127.0.0.1 - - [07/Apr/2026 10:46:24] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Apr/2026 10:46:25] "POST /grade HT